# 03 — ConvNeXt V2 Tiny

This notebook demonstrates the main deep learning model used in the MRI explainability robustness project.

The selected architecture is **ConvNeXt V2 Tiny**, initialized from the pretrained timm checkpoint:

`convnextv2_tiny.fcmae_ft_in22k_in1k`

The notebook demonstrates model initialization and the controlled transfer-learning strategy without presenting development checks as final scientific results.


## 1. Project Setup

The main model uses the same reusable project modules as the baseline pipeline.


In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MRI_XAI_Robustness"
)

SRC_DIR = PROJECT_ROOT / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Project root:', PROJECT_ROOT)


## 2. Reusable Project Modules

Model creation and transfer-learning configuration are implemented in `src/models/main_model.py`.


In [ ]:
from models.main_model import (
    MAIN_MODEL_NAME,
    create_convnextv2_tiny,
    configure_stage1,
    configure_stage2,
)

from train import train_binary_model
from evaluate import collect_participant_predictions
from metrics import calculate_participant_metrics

from utils import (
    DEFAULT_SEED,
    set_random_seed,
    get_device,
    count_parameters,
)

set_random_seed(DEFAULT_SEED)
device = get_device()

print('Random seed:', DEFAULT_SEED)
print('Device:', device)
print('Main model:', MAIN_MODEL_NAME)


## 3. ConvNeXt V2 Tiny Initialization

The model is loaded through `timm` with a single binary classification output.

The selected ConvNeXt V2 Tiny model contains **27,867,265 parameters**.

The 2.5D MRI stacks are supplied as three-channel `3 × 224 × 224` inputs.


In [ ]:
main_model = create_convnextv2_tiny(
    pretrained=True
).to(device)

parameter_counts = count_parameters(
    main_model
)

print(
    'Total parameters:',
    f"{parameter_counts['total_parameters']:,}"
)

assert (
    parameter_counts['total_parameters']
    == 27867265
)

print(
    'PASS: ConvNeXt V2 Tiny initialized correctly.'
)


## 4. Architectural Stages vs. Training Stages

ConvNeXt V2 Tiny contains four major architectural stages, indexed from 0 through 3.

These architectural stages are different from the two transfer-learning stages used in this project:

- **Training Stage 1:** freeze the ConvNeXt V2 backbone and train only the binary classification head.
- **Training Stage 2:** unfreeze architectural stage 3 and the classification head for controlled fine-tuning.


## 5. Transfer-Learning Stage 1 — Head Only

Stage 1 freezes the pretrained ConvNeXt V2 backbone and trains only the binary classification head.

This configuration has **2,305 trainable parameters**.


In [ ]:
main_model = configure_stage1(
    main_model
)

stage1_counts = count_parameters(
    main_model
)

print(
    'Total parameters:',
    f"{stage1_counts['total_parameters']:,}"
)

print(
    'Trainable parameters:',
    f"{stage1_counts['trainable_parameters']:,}"
)

print(
    'Frozen parameters:',
    f"{stage1_counts['frozen_parameters']:,}"
)

print(
    'Trainable percentage:',
    f"{stage1_counts['trainable_percentage']:.4f}%"
)

assert stage1_counts['trainable_parameters'] == 2305
assert stage1_counts['frozen_parameters'] == 27864960

print(
    'PASS: Stage 1 head-only configuration verified.'
)


## 6. Transfer-Learning Stage 2 — Controlled Fine-Tuning

Stage 2 provides a controlled fine-tuning option after head-only training.

The first three ConvNeXt V2 architectural stages remain frozen. Architectural stage 3 and the binary classification head are unfrozen.

This configuration has **15,489,025 trainable parameters**, representing approximately **55.58%** of the model.


In [ ]:
main_model = configure_stage2(
    main_model
)

stage2_counts = count_parameters(
    main_model
)

print(
    'Total parameters:',
    f"{stage2_counts['total_parameters']:,}"
)

print(
    'Trainable parameters:',
    f"{stage2_counts['trainable_parameters']:,}"
)

print(
    'Frozen parameters:',
    f"{stage2_counts['frozen_parameters']:,}"
)

print(
    'Trainable percentage:',
    f"{stage2_counts['trainable_percentage']:.2f}%"
)

assert stage2_counts['trainable_parameters'] == 15489025

print(
    'PASS: Stage 2 fine-tuning configuration verified.'
)


## 7. Weighted Binary Loss

The natural NACC class distribution is retained rather than downsampling the normal group.

The positive-class weight is calculated from the full frozen NACC training partition:

`2394 / 365 ≈ 6.5589`

This same training-partition weight is used during model development rather than recalculating the weight from small development subsets.


In [ ]:
NACC_TRAIN_NORMAL = 2394
NACC_TRAIN_AD = 365

positive_class_weight = (
    NACC_TRAIN_NORMAL
    / NACC_TRAIN_AD
)

pos_weight_tensor = torch.tensor(
    [positive_class_weight],
    dtype=torch.float32,
    device=device,
)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

print(
    'Positive-class weight:',
    f'{positive_class_weight:.4f}'
)


## 8. Differential Learning Rates

For Stage 2, the pretrained final backbone stage uses a smaller learning rate than the newly adapted binary head.

- ConvNeXt V2 architectural stage 3: `1e-5`
- Binary classification head: `1e-4`
- Weight decay: `1e-4`

This allows controlled adaptation of pretrained features while permitting faster adjustment of the classification head.


In [ ]:
stage2_optimizer = torch.optim.AdamW(
    [
        {
            'params': main_model.model.stages[3].parameters(),
            'lr': 1e-5,
        },
        {
            'params': main_model.model.head.parameters(),
            'lr': 1e-4,
        },
    ],
    weight_decay=1e-4,
)

print('Stage 3 learning rate: 1e-5')
print('Head learning rate: 1e-4')
print('Weight decay: 1e-4')

print(
    'PASS: Stage 2 optimizer configured.'
)


## 9. Training Workflow

The main model uses the same reusable training framework as the baseline CNN.

The training function supports weighted binary loss, validation-loss monitoring, checkpointing, CSV logging, and early stopping.

Model-development decisions are based on NACC training and validation data. The official NACC test partition remains untouched during model development.


In [ ]:
# Example training call
#
# history = train_binary_model(
#     model=main_model,
#     train_loader=train_loader,
#     validation_loader=validation_loader,
#     criterion=criterion,
#     optimizer=stage2_optimizer,
#     device=device,
#     checkpoint_path=(
#         PROJECT_ROOT
#         / 'checkpoints'
#         / 'convnextv2_best.pt'
#     ),
#     log_path=(
#         PROJECT_ROOT
#         / 'results'
#         / 'logs'
#         / 'convnextv2_training_log.csv'
#     ),
#     max_epochs=20,
#     patience=5,
# )

print(
    'Reusable ConvNeXt V2 training workflow is ready.'
)


## 10. Participant-Level Evaluation

Each participant contributes five overlapping 2.5D MRI stacks. The five stack-level sigmoid probabilities are averaged to obtain one participant-level probability.

Primary classification metrics are calculated at the participant level.


In [ ]:
# Example participant-level evaluation
#
# stack_predictions, participant_predictions = (
#     collect_participant_predictions(
#         model=main_model,
#         data_loader=validation_loader,
#         device=device,
#     )
# )
#
# metrics = calculate_participant_metrics(
#     participant_predictions,
#     threshold=0.5,
# )
#
# print(metrics)

print(
    'Participant-level evaluation workflow is ready.'
)


## 11. Final Threshold and External Validation

A threshold of 0.5 can be used for development checks, but the final study threshold will be selected using the official NACC validation partition.

Once selected, the threshold will be frozen before:

- final evaluation on the NACC test partition, and
- external validation on the ADNI cohort.

ADNI remains external validation only and is not used for model training, model selection, hyperparameter selection, or threshold selection.


## 12. Explainability and Robustness — Subsequent Stage

After the final classification model is established, the subsequent experimental stage will evaluate the robustness of post-hoc explanations.

Planned explanation methods:

- Grad-CAM
- SHAP

Planned natural perturbations:

- noise
- blur
- small rotations

Planned adversarial perturbations:

- FGSM
- PGD

The primary robustness analysis will focus on cases where the perturbation does not change the predicted class. This allows explanation instability to be examined separately from prediction instability.


## 13. Development-Stage Interpretation

Small development subsets are used only to verify that the pipeline, transfer-learning configurations, and evaluation procedures function correctly.

Performance observed on these small subsets should not be interpreted as final model performance or as evidence that one transfer-learning stage is scientifically superior to another.

Final conclusions require the frozen NACC training, validation, and test workflow followed by ADNI external validation.


## 14. Summary

This notebook demonstrates the reusable main-model pipeline based on ConvNeXt V2 Tiny.

Reusable implementation:

- `src/models/main_model.py`
- `src/dataset.py`
- `src/train.py`
- `src/evaluate.py`
- `src/metrics.py`
- `src/utils.py`

The current repository represents the code-development stage of the project. Explainability, perturbation, robustness, final NACC evaluation, and ADNI external validation are subsequent experimental stages.
